In [1]:
!pip -q install langchain tiktoken chromadb

In [2]:
!pip install openai
!pip install openai langchain-openai



In [3]:
!pip show langchain
!pip install -U langchain-openai


Name: langchain
Version: 0.3.23
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.11/dist-packages
Requires: langchain-core, langchain-text-splitters, langsmith, pydantic, PyYAML, requests, SQLAlchemy
Required-by: langchain-community


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Setting up LangChain


In [5]:
import os

os.environ["OPENAI_API_KEY"] = "key"

In [6]:
!pip install -U langchain-community





In [7]:
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.document_loaders import DirectoryLoader
from pathlib import Path


In [8]:
!pip install pypdf
!pip install -U langchain-chromadb



from langchain.document_loaders import PyPDFLoader

ERROR: Could not find a version that satisfies the requirement langchain-chromadb (from versions: none)
ERROR: No matching distribution found for langchain-chromadb


In [9]:
pdf_dir = "/content/drive/MyDrive/fact-check/data/Clinical Files"
pdf_dir = Path(pdf_dir)
docs = []
for pdf_path in pdf_dir.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())

print(f"Loaded {len(docs)} documents.")

Loaded 85 documents.


In [10]:
!pip install --upgrade openai


In [11]:
import openai
print(openai.OpenAI)

<class 'openai.OpenAI'>


In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(docs)
print(f"Split into {len(texts)} chunks.")

Split into 617 chunks.


## create the DB

In [13]:
persist_directory = "/content/drive/MyDrive/fact-check/db"
embedding = OpenAIEmbeddings()
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name="factcheck_docs"
)
vectordb.persist()
print(f"Persisted {vectordb._collection.count()} vectors to {persist_directory}")

Persisted 1851 vectors to /content/drive/MyDrive/fact-check/db


<ipython-input-13-a166d2d01fed>:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [14]:
# Now we can load the persisted database from disk, and use it as normal.
vectordb = Chroma(persist_directory=persist_directory,
                  embedding_function=embedding)

<ipython-input-14-39e6cb1c6266>:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory,


In [16]:

vectordb = Chroma(
    persist_directory="/content/drive/MyDrive/fact-check/db",
    embedding_function=embedding,
    collection_name="factcheck_docs",
)

# 1) How many vectors?
count = vectordb._collection.count()
print(f"✅ There are {count} vectors in the store.")

# 2) Peek at the first 3 documents + metadata
raw = vectordb._collection.get(
    limit=3,
    include=["documents", "metadatas"]
)
for doc, meta in zip(raw["documents"], raw["metadatas"]):
    print(f"\n— source: {meta}\n  text: {doc[:200]}…")

# 3) Do a quick similarity‐search:
results = vectordb.similarity_search("What is Flublok?", k=2)
for r in results:
    print("\n", r.page_content[:200], "…")

✅ There are 1851 vectors in the store.

— source: {'page_label': 'i', 'subject': 'ACIP recommendations for use of seasonal influenza vaccines during the 2022–23 season in the U.S.', 'title': 'Prevention and Control of Seasonal Influenza with Vaccines: Recommendations of the Advisory Committee on Immunization Practices — United States, 2022–23 Influenza Season', 'moddate': '2022-08-18T16:57:26-04:00', 'trapped': '/False', 'producer': 'Adobe PDF Library 16.0.7', 'page': 0, 'creationdate': '2022-08-18T16:46:16-04:00', 'source': '/content/drive/MyDrive/fact-check/data/Clinical Files/Grohskopf et al. (2023).pdf', 'author': 'Centers for Disease Control and Prevention', 'creator': 'Adobe InDesign 17.3 (Windows)', 'total_pages': 32}
  text: Morbidity and Mortality Weekly Report
Recommendations and Reports / Vol. 71 / No. 1 August 26, 2022 
U.S. Department of Health and Human Services
Centers for Disease Control and Prevention
Prevention …

— source: {'source': '/content/drive/MyDrive/fact-chec

In [27]:
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.0)
from langchain.schema import SystemMessage, HumanMessage

In [18]:
claims = [
    { "claim": "Flublok ensures identical antigenic match with WHO- and FDA-selected flu strains." },
    { "claim": "Flublok contains 3x the hemagglutinin (HA) antigen content of standard-dose flu vaccines, which has been linked to greater immunogenicity vs standard-dose flu vaccines." },
    { "claim": "Cell- and egg-based flu vaccines have the potential to develop mutations during production, which may reduce their effectiveness." },
    { "claim": "Recombinant technology leads to a broader immune response that may provide cross-protection, even in a mismatch season." },
    { "claim": "Vaccination with a higher-dose recombinant flu vaccine may induce a more robust antibody response than egg-based standard-dose vaccines." },
    { "claim": "Flublok (quadrivalent) was evaluated in the pivotal trial against Fluarix (quadrivalent standard-dose vaccine)." },
    { "claim": "Flublok is produced using a novel production platform in which recombinant HA is expressed in insect cells using a baculovirus expression vector system (BEVS)." },
    { "claim": "Recombinant HA antigens produced using BEVS have been shown to induce significantly higher levels of broadly cross-reactive antibodies against highly conserved regions of HA compared with egg-derived vaccines." },
    { "claim": "Flublok contains 45 micrograms (mcg) of HA per strain vs 15 mcg of HA per strain in a standard-dose influenza vaccine." }
]

In [40]:
# Cell Y — ToRAG function, corrected and self‑contained

import re
from typing import Dict, List
from langchain.schema import SystemMessage, HumanMessage
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import Chroma


llm = ChatOpenAI(model_name="gpt-4o", temperature=0.0)

def torag_for_claim(
    claim: str,
    num_initial: int = 3,
    k: int = 5
) -> Dict:
    """
    Perform a single ToRAG pass for one claim:
      1) Generate N initial investigative questions
      2) For each question: retrieve top-k docs + answer from the LLM
      3) Let the LLM pick the most helpful Q/A pair

    Returns a dict with the claim, branches, chosen index, the best branch,
    and the full explanation text from the LLM.
    """
    # 1) Generate initial questions
    q_messages = [
        SystemMessage(content="You are a fact‑checking assistant."),
        HumanMessage(content=(
            f"Claim: {claim}\n"
            f"Generate exactly {num_initial} numbered, focused questions to investigate this claim."
        ))
    ]
    q_resp = llm.generate([q_messages])
    raw_q = q_resp.generations[0][0].message.content.strip()
    # split on lines, take first N non-empty
    questions = [line.strip() for line in raw_q.split("\n") if line.strip()][:num_initial]

    branches: List[Dict] = []
    # 2) For each question: retrieve & answer
    for q in questions:
        docs = vectordb.similarity_search(q, k=k)
        context = "\n".join(f"- {d.page_content}" for d in docs)

        a_messages = [
            SystemMessage(content="You are an evidence‑based assistant."),
            HumanMessage(content=(
                f"Question: {q}\n\n"
                f"Context:\n{context}\n\n"
                "Answer succinctly, quoting from the context."
            ))
        ]
        a_resp = llm.generate([a_messages])
        answer = a_resp.generations[0][0].message.content.strip()

        branches.append({
            "question": q,
            "answer":   answer,
            "evidence": [d.page_content for d in docs]
        })

    # 3) Ask the LLM to pick the best branch
    pick_text = "\n\n".join(
        f"{i+1}. Q: {b['question']}\n   A: {b['answer']}"
        for i, b in enumerate(branches)
    )
    pick_messages = [
        SystemMessage(content="You are a fact‑checking assistant selecting the most helpful Q/A."),
        HumanMessage(content=(
            f"Here are the Q/A pairs:\n\n{pick_text}\n\n"
            f"Which number (1–{len(branches)}) is most helpful for verifying the claim? "
            "You can explain your reasoning, but please include the chosen number in your answer."
        ))
    ]
    pick_resp = llm.generate([pick_messages])
    reply = pick_resp.generations[0][0].message.content.strip()

    # pull out the first integer we find
    match = re.search(r'\b([1-9][0-9]*)\b', reply)
    if not match:
        raise ValueError(f"Couldn't parse branch index from:\n{reply!r}")
    idx = int(match.group(1))

    return {
        "claim":             claim,
        "branches":          branches,
        "best_branch_index": idx - 1,
        "best_branch":       branches[idx - 1],
        "pick_explanation":  reply
    }


In [41]:
# run it over your inline list of claims, save & display one sample
import json, pprint

import re
claims = [
    { "claim": "Flublok ensures identical antigenic match with WHO- and FDA-selected flu strains." },
    { "claim": "Flublok contains 3x the hemagglutinin (HA) antigen content of standard-dose flu vaccines, which has been linked to greater immunogenicity vs standard-dose flu vaccines." },
    { "claim": "Cell- and egg-based flu vaccines have the potential to develop mutations during production, which may reduce their effectiveness." },
    { "claim": "Recombinant technology leads to a broader immune response that may provide cross-protection, even in a mismatch season." },
    { "claim": "Vaccination with a higher-dose recombinant flu vaccine may induce a more robust antibody response than egg-based standard-dose vaccines." },
    { "claim": "Flublok (quadrivalent) was evaluated in the pivotal trial against Fluarix (quadrivalent standard-dose vaccine)." },
    { "claim": "Flublok is produced using a novel production platform in which recombinant HA is expressed in insect cells using a baculovirus expression vector system (BEVS)." },
    { "claim": "Recombinant HA antigens produced using BEVS have been shown to induce significantly higher levels of broadly cross-reactive antibodies against highly conserved regions of HA compared with egg-derived vaccines." },
    { "claim": "Flublok contains 45 micrograms (mcg) of HA per strain vs 15 mcg of HA per strain in a standard-dose influenza vaccine." }
]
results = {"claims": []}
for idx, item in enumerate(claims, 1):
    print(f" Processing claim {idx}/{len(claims)}")
    out = torag_for_claim(item["claim"])
    results["claims"].append(out)

# 3) Write everything out
with open("torag2.json", "w") as f:
    json.dump(results, f, indent=2)

# 4) Sanity‑check: print how many we got and the first result
print(f"\n Completed ToRAG for {len(results['claims'])} claims.")
print("\n--- Sample for claim #1 ---")
pprint.pp(results["claims"][0])



 Processing claim 1/9
 Processing claim 2/9
 Processing claim 3/9
 Processing claim 4/9
 Processing claim 5/9
 Processing claim 6/9
 Processing claim 7/9
 Processing claim 8/9
 Processing claim 9/9

 Completed ToRAG for 9 claims.

--- Sample for claim #1 ---
{'claim': 'Flublok ensures identical antigenic match with WHO- and '
          'FDA-selected flu strains.',
 'branches': [{'question': '1. What is the process by which Flublok is '
                           'developed, and how does it ensure an antigenic '
                           'match with the flu strains selected by the WHO and '
                           'FDA?',
               'answer': 'Flublok is developed using a process that does not '
                         'involve chicken eggs, which helps avoid mutations in '
                         'the hemagglutinin protein that can occur during '
                         'egg-based manufacturing. This process results in a '
                         'recombinant hemagglutinin 

In [53]:
pdf_dir = Path("/content/drive/MyDrive/fact-check/data/Clinical Files")

# 2) load each PDF (one Document per page)
docs = []
for pdf_path in pdf_dir.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())
print(f"Loaded {len(docs)} pages from PDF files")

# 3) split into ~1 000‑char chunks with 200‑char overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = []
for pdf_path in pdf_dir.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()  # one Document per page

    chunks = text_splitter.split_documents(pages)
    for chunk in chunks:
        chunk.metadata["source_file"] = str(pdf_path.name)
        chunk.metadata["page"] = chunk.metadata.get("page", None)
        texts.append(chunk)

print(f"Split into {len(texts)} chunks")

# 4) choose a new persist directory so this index stays separate
new_index_dir = "/content/drive/MyDrive/fact-check/db_claims"

# 5) create & persist the Chroma index
embedding = OpenAIEmbeddings()
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embedding,
    persist_directory=new_index_dir,
    collection_name="claims_index"
)
vectordb.persist()
print(f"✅ Persisted {vectordb._collection.count()} vectors to {new_index_dir}")

# 6) reload it for querying in your ToRAG cell
vectordb = Chroma(
    persist_directory=new_index_dir,
    embedding_function=embedding,
    collection_name="claims_index"
)
print(f"🔍 Loaded vector store with {vectordb._collection.count()} vectors")

Loaded 85 pages from PDF files
Split into 617 chunks
✅ Persisted 1234 vectors to /content/drive/MyDrive/fact-check/db_claims
🔍 Loaded vector store with 1234 vectors


In [52]:
persist_directory = "/content/drive/MyDrive/fact-check/db_claims"
embedding = OpenAIEmbeddings()
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding,
    collection_name="claims_index"
)



In [54]:
def torag_for_claim2(
    claim: str,
    num_initial: int = 3,
    k: int = 5
) -> Dict:
    """
    ToRAG for one claim, returning:
      - the initial questions
      - an answer + evidence list (with source_file/page) per question
      - which branch the LLM picked
    """
    # Step 1: Generate initial questions
    q_messages = [
        SystemMessage(content="You are a fact‑checking assistant."),
        HumanMessage(content=(
            f"Claim: {claim}\n"
            f"Generate exactly {num_initial} distinct, numbered questions to investigate this claim."
        ))
    ]
    q_resp = llm.generate([q_messages])
    raw_q = q_resp.generations[0][0].message.content.strip()
    all_lines = [ln.strip() for ln in raw_q.splitlines() if ln.strip()]
    questions = all_lines[:num_initial]

    branches: List[Dict] = []
    for q in questions:
        q_text = re.sub(r'^\d+[\.\)]\s*', '', q).strip()
        docs = vectordb.similarity_search(q_text, k=k)

        # build evidence entries
        evidence = []
        for d in docs:
            evidence.append({
                "source_file": d.metadata.get("source_file", "<unknown>"),
                "page":        d.metadata.get("page", None),
                "text":        d.page_content
            })

        # Step 2: Answer the question based on evidence
        a_messages = [
            SystemMessage(content="You are an evidence‑based assistant."),
            HumanMessage(content=(
                f"Question: {q_text}\n\n"
                "Context (snippets from clinical docs):\n" +
                "\n".join(f"- {ev['text']}" for ev in evidence) +
                "\n\nAnswer succinctly, quoting directly from the context."
            ))
        ]
        a_resp = llm.generate([a_messages])
        answer = a_resp.generations[0][0].message.content.strip()

        branches.append({
            "question": q_text,
            "answer":   answer,
            "evidence": evidence
        })

    # Step 3: Let LLM choose the best Q/A
    pick_text = "\n\n".join(
        f"{i+1}. Q: {b['question']}\n   A: {b['answer']}"
        for i, b in enumerate(branches)
    )
    pick_messages = [
        SystemMessage(content="You are a fact‑checking assistant selecting the most helpful Q/A."),
        HumanMessage(content=(
            "Here are the Q/A pairs:\n\n" + pick_text +
            f"\n\nWhich number (1–{len(branches)}) is most helpful for verifying the claim? "
            "You can explain your reasoning, but please include the chosen number."
        ))
    ]
    pick_resp = llm.generate([pick_messages])
    pick_reply = pick_resp.generations[0][0].message.content.strip()

    m = re.search(r'\b([1-9][0-9]*)\b', pick_reply)
    if not m:
        raise ValueError(f"Couldn't parse branch index from:\n{pick_reply!r}")
    chosen = int(m.group(1))

    return {
        "claim":             claim,
        "questions":         questions,
        "branches":          branches,
        "best_branch_index": chosen - 1,
        "best_branch":       branches[chosen - 1],
        "pick_explanation":  pick_reply
    }


In [55]:
claims = [
    "Flublok ensures identical antigenic match with WHO- and FDA-selected flu strains.",
    "Flublok contains 3x the hemagglutinin (HA) antigen content of standard-dose flu vaccines, which has been linked to greater immunogenicity vs standard-dose flu vaccines.",
    "Cell- and egg-based flu vaccines have the potential to develop mutations during production, which may reduce their effectiveness.",
    "Recombinant technology leads to a broader immune response that may provide cross-protection, even in a mismatch season.",
    "Vaccination with a higher-dose recombinant flu vaccine may induce a more robust antibody response than egg-based standard-dose vaccines.",
    "Flublok (quadrivalent) was evaluated in the pivotal trial against Fluarix (quadrivalent standard-dose vaccine).",
    "Flublok is produced using a novel production platform in which recombinant HA is expressed in insect cells using a baculovirus expression vector system (BEVS).",
    "Recombinant HA antigens produced using BEVS have been shown to induce significantly higher levels of broadly cross-reactive antibodies against highly conserved regions of HA compared with egg-derived vaccines.",
    "Flublok contains 45 micrograms (mcg) of HA per strain vs 15 mcg of HA per strain in a standard-dose influenza vaccine."
]

#  Run for all claims
results = {"claims": []}
for i, c in enumerate(claims, 1):
    print(f"🔎 Processing claim {i}/{len(claims)}…")
    out = torag_for_claim2(c)
    results["claims"].append(out)

#  Save results
with open("torag4.json", "w") as f:
    json.dump(results, f, indent=2)

#  Sanity check
print(f"\n Completed ToRAG for {len(results['claims'])} claims.")
print("\n--- Sample for claim #1 ---")
pprint.pprint(results["claims"][0])

🔎 Processing claim 1/9…
🔎 Processing claim 2/9…
🔎 Processing claim 3/9…
🔎 Processing claim 4/9…
🔎 Processing claim 5/9…
🔎 Processing claim 6/9…
🔎 Processing claim 7/9…
🔎 Processing claim 8/9…
🔎 Processing claim 9/9…

 Completed ToRAG for 9 claims.

--- Sample for claim #1 ---
{'best_branch': {'answer': 'Flublok is developed using a process that avoids '
                           'the use of chicken eggs, which helps prevent '
                           'mutations in the hemagglutinin protein that can '
                           'occur during egg-based manufacturing. This process '
                           'results in a recombinant hemagglutinin protein '
                           'that is "genetically identical to that in the '
                           'selected strain." This ensures an antigenic match '
                           'with the flu strains selected by the WHO and FDA. '
                           'Additionally, Flublok contains "three times the '
                   